|<h2>Course:</h2>|<h1><a href="https://udemy.com/course/dullms_x/?couponCode=202508" target="_blank">A deep understanding of AI language model mechanisms</a></h1>|
|-|:-:|
|<h2>Part 2:</h2>|<h1>Large language models<h1>|
|<h2>Section:</h2>|<h1>Fine-tune pretrained models<h1>|
|<h2>Lecture:</h2>|<h1><b>Fine-tune a pretrained GPT2<b></h1>|

<br>

<h5><b>Teacher:</b> Mike X Cohen, <a href="https://sincxpress.com" target="_blank">sincxpress.com</a></h5>
<h5><b>Course URL:</b> <a href="https://udemy.com/course/dullms_x/?couponCode=202508" target="_blank">udemy.com/course/dullms_x/?couponCode=202508</a></h5>
<i>Using the code without the course may lead to confusion or errors.</i>

In [1]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoModelForCausalLM,GPT2Tokenizer
import requests

# vector plots
import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

In [2]:
# load pretrained GPT-2 model and tokenizer
gpt2 = AutoModelForCausalLM.from_pretrained('gpt2')
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

c:\Users\alex huang2\Udemy\A Deep Understanding of AI Large Language Model Mechanisms\LLM_course\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\alex huang2\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [3]:
# hyperparameters
seq_len    = 256 # max sequence length
batch_size =  16

# use GPU
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [ ]:
# tokenize the text
# Gulliver's travels :)
text = requests.get('https://www.gutenberg.org/cache/epub/829/pg829.txt').text

# the old way:
gtTokens = torch.tensor( tokenizer.encode(text),dtype=torch.long )  
            # tokenizer.encode(text) returns a python list
            # torch.tensor(..., dtype=torch.long) converts a python list to pytorch tensor 
            # This together returns a tensor of shape [158340]
print(gtTokens.shape)

# a better way ;)
gtTokens = tokenizer.encode(text,return_tensors='pt')   
            # This returns a pythorch tensor directly
            # This returns a tensor of shape [1, 158340] which is a 2-dimensional tensor.
print(gtTokens.shape)

# but the rest of the code is setup for dimensionless tensors
gtTokens = gtTokens[0]      # Note gtTokens has shape [1, 158340]
                            # This line extract the first row which is a 1-dimentional tensor with shape [158340].
print(gtTokens.shape)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (158340 > 1024). Running this sequence through the model will result in indexing errors


torch.Size([158340])
torch.Size([1, 158340])
torch.Size([158340])


In [ ]:
# find the most frequent 100 tokens
uniq,counts = np.unique(gtTokens,return_counts=True)    # Find unique tokens and return the counts.
freqidx = np.argsort(counts)[::-1]                      # Short them.
top100 = uniq[freqidx[:100]]                            # Get the first 100 most common tokens.

for t in top100:
  print(f'Token {t:5} appears {torch.sum(gtTokens==t):4} times and is "{tokenizer.decode(t)}"')

Token    11 appears 10060 times and is ","
"oken   201 appears 9923 times and is "
Token   198 appears 9923 times and is "
"
Token   262 appears 5393 times and is " the"
Token   286 appears 3779 times and is " of"
Token   290 appears 3468 times and is " and"
Token   284 appears 3356 times and is " to"
Token    13 appears 3006 times and is "."
Token   314 appears 2657 times and is " I"
Token   257 appears 2431 times and is " a"
Token   287 appears 1900 times and is " in"
Token    26 appears 1562 times and is ";"
Token   616 appears 1474 times and is " my"
Token   373 appears 1199 times and is " was"
Token   326 appears 1061 times and is " that"
Token   502 appears  964 times and is " me"
Token   351 appears  941 times and is " with"
Token   355 appears  920 times and is " as"
Token   465 appears  823 times and is " his"
Token   416 appears  794 times and is " by"
Token   329 appears  776 times and is " for"
Token   543 appears  774 times and is " which"
Token   447 appears  734 times an

In [7]:
# move the model to the GPU
gpt2 = gpt2.to(device)

In [ ]:
# check out some text   ( Generate some text )
prompt = 'I cannot believe that'
in2gpt = tokenizer.encode(prompt,return_tensors='pt').to(device)

output = gpt2.generate(in2gpt,max_length=100,pad_token_id=50256,do_sample=True).cpu()
print(tokenizer.decode(output[0]))

I cannot believe that Trump has a chance to win.

"Look at the women, look at the issues. Look, we have no idea who they are. All we know is 'You're a man.'" — Michelle Malkin


"They might need to start doing some of that work for the government. They might need to start doing some of that work for the government. We need to go after ISIS, we need to go after the Islamic State and then we can get back


# Quantify the frequency of GT frequent tokens in GPT's output

In [ ]:
numreps =  10 # number of random repetitions         ( number of batches )
numtoks = 100 # output length

# random starting tokens
randstarts = torch.randint(tokenizer.vocab_size,(numreps,1)).to(device)
      # This will be a tensor of shape [numreps, 1] which is [10, 1] 

# generate some data
out = gpt2.generate(
  randstarts,               # randstarts has shape [10, 1]
  max_length = numtoks+1,   # numtoks + 1 because the first token is from input but it is included in output.
  min_length = numtoks+1,
  do_sample  = True,
  bad_words_ids = [tokenizer.encode(tokenizer.eos_token)],
  pad_token_id = tokenizer.encode(tokenizer.eos_token)[0]
).cpu()

print(out,'\n')

for o in out:
  print('\n*** Next batch of output:')
  print(tokenizer.decode(o))

[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


tensor([[18875,   661,   547,  ...,   749,  1176,    11],
        [19739,   290,   534,  ...,  9616,   345,  3853],
        [23548,    11,   317,  ...,  2599,   317,  7208],
        ...,
        [34993,   319,   262,  ...,   262, 12056,     6],
        [34355,     8,   625,  ...,   284,   766,   319],
        [28007,   338, 15067,  ...,   277,   532,   257]]) 


*** Next batch of output:
 literal people were a bunch of idiots. I mean I didn't care. The only reason I left behind all these stupid people was because I wanted to change a character. And I would have wanted to read everyone who hated me. It was because the book wasn't going to last, that we had already found the right people for the stories. Why did there was so much hype around this book. Because even when I read the most powerful characters in my life, the one who had the most power,

*** Next batch of output:
 Curse and your next quest, "Cannon, Cannon, Flame" will drop the first recipe for these.

The second recipe was j

In [10]:
# calculate and report the percentage
percentFreqTokens_pre = np.mean(100*np.isin(out[:,1:],top100).flatten())
print(f"Gulliver's travels common tokens appeared in {percentFreqTokens_pre}% of new tokens.")

Gulliver's travels common tokens appeared in 45.1% of new tokens.


# Fine-tune the model

In [ ]:
# create the optimizer functions (note the small learning rate)
optimizer = torch.optim.AdamW(gpt2.parameters(), lr=5e-5, weight_decay=.01)
      # The learning rate lr=5e-5 which is small because you don't want to overwrite the word
      # knowledge and the grammar and syntax and structure and patterns in language that these 
      # pre-trained models have already learned.
# Note: don't need the loss function here, because it's calculated internally in the model (thanks HF :D )

In [ ]:
num_samples = 1234

# initialize losses
train_loss = np.zeros(num_samples)

for sampli in range(num_samples):

  # get a batch of data
  ix = torch.randint(len(gtTokens)-seq_len,size=(batch_size,))
  X  = gtTokens[ix[:,None] + torch.arange(seq_len)]

  # move data to GPU
  X = X.to(device)

  # clear previous gradients
  gpt2.zero_grad()

  # forward pass (Hugging Face shifts X internally to get y)
  outputs = gpt2(X,labels=X)    # When setting labels=X which is same as the inputs, then hugging face
                                #   models internally will will take all of these input tokens and shift
                                #   them by one and use those as the targets.
                                # Also, inside the model, it's going to make sur that it's using the 
                                #   negative log or log softmax values.
  loss = outputs.loss   # This is the lost. We do not need to have our own separate loss function.

  # backprop
  loss.backward()
  optimizer.step()

  # store the per-sample loss
  train_loss[sampli] = loss.item()

  # update progress display
  if sampli%77==0:
    print(f'Sample {sampli:4}/{num_samples}, train loss: {train_loss[sampli]:.4f}')

In [ ]:
# plot the losses
plt.figure(figsize=(8,4))
plt.plot(train_loss,'k',markersize=8)

plt.gca().set(xlabel='Data sample',ylabel='Train loss',xlim=[-1,num_samples])
plt.show()

In [ ]:
# Qualtative assessment
prompt = 'I cannot believe that'
in2gpt = tokenizer.encode(prompt,return_tensors='pt').to(device)

output = gpt2.generate(in2gpt,max_length=100,pad_token_id=50256)
print(tokenizer.decode(output[0]))

# Calculate percentage of GT tokens generated

In [ ]:
# random starting tokens
randstarts = torch.randint(tokenizer.vocab_size,(numreps,1)).to(device)

# generate some data
out = gpt2.generate(
  randstarts,
  max_length = numtoks+1,
  min_length = numtoks+1,
  do_sample  = True,
  bad_words_ids = [tokenizer.encode(tokenizer.eos_token)],
  pad_token_id = tokenizer.encode(tokenizer.eos_token)[0]
).cpu()


for o in out:
  print('\n*** Next batch of output:')
  print(tokenizer.decode(o))

In [ ]:
# calculate and report the percentage
percentFreqTokens_pst = np.mean(100*np.isin(out[:,1:],top100).flatten())

print(f'Common GT tokens usage went from {percentFreqTokens_pre:.2f}% to {percentFreqTokens_pst:.2f}% after fine-tuning.')